In [ ]:
import json, base64, re
import pickle as pkl

from typing import TypedDict, Dict, Any, List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

from langchain.globals import set_debug, set_verbose
set_debug(False)
set_verbose(False)

In [6]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

## HELPER FUNC

In [11]:
def add_trace(state: dict, *, agent: str, role: str, payload):
    if "trace" not in state:
        state["trace"] = []
    state["trace"].append({
        "agent": agent,
        "role": role,
        "payload": payload,
    })
    
def extract_first_json(text: Any) -> Dict[str, Any]:
    """
    Extract the first JSON object from `text`.
    - Handles ```json fences.
    - Finds the first balanced {...} with brace counting.
    - Tries lightweight repairs (auto-closing braces).
    - Always returns a dict: parsed JSON or {"_raw": ..., "_reason": ...}.
    """
    # Fast paths for non-strings
    if isinstance(text, dict):
        return text
    if text is None:
        return {"_raw": "", "_reason": "none_input"}
    if not isinstance(text, str):
        return {"_raw": str(text), "_reason": f"not_a_string:{type(text).__name__}"}

    original = text
    s = text.strip()
    
    # Strip leading ``` or ```json fences and a trailing closing fence
    s = re.sub(r"^```[\w-]*\s*\n", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\n```[\s\t]*$", "", s)

    # 0) Whole-string JSON
    try:
        val = json.loads(s)
        return val if isinstance(val, dict) else {"_raw": s, "_reason": "json_root_not_object"}
    except Exception:
        pass

    # 1) Find first balanced { ... }
    start = s.find("{")
    if start == -1:
        return {"_raw": s, "_reason": "no_brace_found"}

    depth = 0
    in_string = False
    escape = False
    end = None

    for i, ch in enumerate(s[start:], start=start):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break

    candidate = s[start:(end + 1) if end is not None else len(s)]

    # 2) Parse balanced candidate
    if end is not None:
        try:
            val = json.loads(candidate)
            return val if isinstance(val, dict) else {"_raw": candidate, "_reason": "json_root_not_object"}
        except Exception as e:
            return {"_raw": candidate, "_reason": f"parse_error_balanced:{type(e).__name__}"}

    # 3) Try auto-closing braces (only if not inside a string)
    if not in_string and depth > 0:
        repaired = candidate + ("}" * depth)
        try:
            val = json.loads(repaired)
            return val if isinstance(val, dict) else {"_raw": repaired, "_reason": "json_root_not_object"}
        except Exception as e:
            return {"_raw": repaired, "_reason": f"parse_error_repaired:{type(e).__name__}"}

    # 4) Last resort
    return {"_raw": original, "_reason": "unbalanced_in_string_or_unknown"}

## MODEL SETUP

In [12]:
observer_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0.0,
    top_p=0.95
)

triage_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0.3,
    top_p=0.95
)

diag_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0.7,
    top_p=0.95
)

In [13]:
import base64

def encode_image(image_path):
    """Encode a local image to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')
    
def invoke_llm(model, SYSTEM_PROMPT, user_message):

    messages =[
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message)
    ]

    response = model.invoke(messages)
    return extract_first_json(response.text())

## PROMPT

In [50]:
LAB_TECHNICIAN = """
You are a meticulous AI Lab Technician. Your role is to consolidate all available patient and image data into a single, structured report. You will perform an objective visual analysis of the provided image, but you MUST NOT make a diagnosis or infer any disease. Your task is to report only the facts.

Instructions:
  1. Receive the patient's metadata (age, sex, location, melanocytic status).
  2. Receive the dermatoscopic image.
  3. For the `dermoscopic_features` section, identify ALL relevant features you observe in the image. For each feature, create a separate object in the list containing its name and a detailed description.
  4. Include common features like pigment patterns, vascular structures, and surface characteristics. If a common feature (like a pigment network) is absent, do not create an object for it.

Output Format:
You must output ONLY a single, valid JSON object with the following structure.
{
  "patient_data": {
    "age": <age>,
    "sex": "<sex>",
    "location": "<location>"
  },
  "lesion_metadata": {
    "is_melanocytic": <true | false>
  },
  "visual_summary": {
    "symmetry": "<Symmetrical | Asymmetrical>",
    "colors": ["<color1>", "<color2>", "..."],
    "border_characteristics": "<Well-defined | Ill-defined | Irregular | Fading>"
  },
  "dermoscopic_features": [
    {
      "feature_name": "<The name of the first observed feature, e.g., 'Pigment Globules'>",
      "description": "<A detailed description of this feature, including its color, size, and location.>"
    },
    {
      "feature_name": "<The name of the second observed feature, e.g., 'Surface Texture'>",
      "description": "<A detailed description of the surface and texture, e.g., 'The surface appears smooth and slightly waxy in the center, with some fine scaling at the periphery.'>"
    },
    {
      "feature_name": "<The name of the third observed feature, e.g., 'Vascular Pattern'>",
      "description": "<A detailed description of the vessels, e.g., 'Fine linear vessels are visible in the reddish areas, consistent with telangiectasias.'>"
    }
  ]
}
"""

In [40]:
image_path = "../dataset/test/ISIC_0033626.jpg"
image_data = encode_image(image_path)

patient_data = {
    "age": 35,
    "sex": "female",
    "lesion_location": "lower extremity",
    # "is_melanocytic": False
}

user_message = [
    {
        "type": "text",
        "text": "Process the following case and generate the initial report.\n"
    },
    {
        "type": "text",
        "text": f"Patient Data:\n {patient_data}\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
    }
    
]

lab_tech_llm  = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0,
    top_p=0.95
)

image_features = invoke_llm(lab_tech_llm, LAB_TECHNICIAN, user_message)
print(image_features)

{'patient_data': {'age': 35, 'sex': 'female', 'location': 'lower extremity'}, 'visual_summary': {'symmetry': 'Asymmetrical', 'colors': ['brown', 'light brown', 'reddish-pink', 'whitish'], 'border_characteristics': 'Ill-defined'}, 'dermoscopic_features': [{'feature_name': 'Homogeneous Pigmentation', 'description': 'The lesion displays diffuse, structureless pigmentation. A central reddish-pink hue transitions to a light brown and brown coloration towards the periphery, without a clear pigment network.'}, {'feature_name': 'Whitish Structureless Area', 'description': 'A central, somewhat irregular whitish area is present, lacking discernible specific structures or patterns.'}, {'feature_name': 'Reddish Areas', 'description': 'A prominent reddish-pink coloration is observed centrally, suggesting increased vascularity or inflammation, though individual vessel morphology is not clearly resolved at this magnification.'}, {'feature_name': 'Ill-defined Border', 'description': 'The periphery of 

In [45]:
TRIAGE_PROMPT = """
You are an expert AI Triage Specialist. Your task is to determine the broad family of a skin lesion. You will receive an initial report containing metadata AND the original image.

Instructions:
1.  First, check the `is_melanocytic` flag in the report. This is your primary guide.
2.  IF `is_melanocytic` is `true`, classify the family as "Melanocytic Proliferations".
3.  IF `is_melanocytic` is `false`, you must analyze the provided image directly, using the `visual_summary` as context, to classify the lesion into one of the following NON-melanocytic families:
    - "Keratinocyte Tumors (Benign)"
    - "Keratinocyte Tumors (Malignant/Pre-malignant)"
    - "Fibrohistiocytic Tumors"
4.  Your visual analysis should confirm the classification. For example, if you see features of a BCC in the image, that strongly supports the 'Malignant' category.

Output Format:
{
  "disease_family": "<The name of the classified disease family>",
  "reasoning": "<Your reasoning for the classification>"
}
"""

In [43]:
TEST_TRIAGE = """
You are an expert AI Triage Specialist. Your task is to determine the broad family of a skin lesion by following a rigorous, step-by-step reasoning process.

# --- Chain-of-Thought Triage Framework ---
Before providing your final JSON output, you MUST internally follow these steps:
1.  **Identify Salient Features:** Review the initial report and the image. Summarize the 2-3 most diagnostically important visual features (e.g., "asymmetrical and multicolored," "pink, scaly patch," "stuck-on appearance with cysts").
2.  **Make the Critical Distinction:** Based on the features, address the most important question: Is the lesion more likely melanocytic or non-melanocytic? Justify this decision, paying close attention to the presence and characteristics of a pigment network (or its absence).
3.  **Evaluate Plausible Families:** Based on your decision in Step 2, evaluate the most likely family.
    -   If you concluded it is **melanocytic**, the family is "Melanocytic Proliferations".
    -   If you concluded it is **non-melanocytic**, briefly weigh the evidence to decide which of the non-melanocytic families is the best fit.
4.  **Conclude Classification:** State your final family classification based on the evidence.

# --- Possible Disease Families ---
- "Melanocytic Proliferations"
- "Keratinocyte Tumors (Benign)"
- "Keratinocyte Tumors (Malignant/Pre-malignant)"
- "Fibrohistiocytic Tumors"

# --- Final Output ---
After completing your internal chain-of-thought analysis, provide your final answer in the following JSON format ONLY. The reasoning in the JSON should be a concise summary of your thought process, especially your justification from Step 2.
{
  "disease_family": "<Your final classified disease family>",
  "reasoning": "<A concise summary of your chain of thought, explaining how you arrived at the final family classification.>"
}
"""

In [46]:
user_message = [
    {
        "type": "text",
        "text": "Based on the following report AND the provided image, perform triage and classify the disease family.\n"
    },
    {
        "type": "text",
        "text": "Skin Lesion Image:\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
    },
    {
        "type": "text",
        "text": f"\nInitial Report:\n{json.dumps(image_features, ensure_ascii=False)}"
    }
    
] 

triage_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    top_p=0.95
)

triage_result = invoke_llm(triage_llm, TRIAGE_PROMPT, user_message)
print(triage_result)

{'disease_family': 'Fibrohistiocytic Tumors', 'reasoning': "The initial report does not contain an `is_melanocytic` flag, so the classification is based on visual analysis and dermoscopic features. The key dermoscopic features described are 'Homogeneous Pigmentation' (without a pigment network), a 'Whitish Structureless Area' centrally, and 'Reddish Areas' with an 'Ill-defined Border'. This combination of features, particularly the central whitish scar-like area surrounded by homogeneous pigmentation and reddish hues, is highly characteristic of a Dermatofibroma, which belongs to the Fibrohistiocytic Tumors family. While some features might overlap with other non-melanocytic lesions (e.g., pigmented Basal Cell Carcinoma), the central whitish structureless area is a strong indicator for Dermatofibroma."}


In [49]:
DIAGNOSIS_PROMPT = """
You are a specialist AI Dermoscopist. Your task is to provide a final, specific diagnosis. You will receive an initial report, a `disease_family` classification, AND the original image.

Hierarchical Diagnostic Rules:
-  IF `disease_family` is "Melanocytic Proliferations": Your diagnosis MUST be either `Melanoma` or `Nevus`.
-  IF `disease_family` is "Keratinocyte Tumors (Benign)": Your diagnosis MUST be `Pigmented Benign Keratosis`.
-  IF `disease_family` is "Keratinocyte Tumors (Malignant/Pre-malignant)": Your diagnosis MUST be one of: `Basal Cell Carcinoma`, `Squamous Cell Carcinoma`, or `Actinic Keratosis`.
-  IF `disease_family` is "Fibrohistiocytic Tumors": Your diagnosis MUST be `Dermatofibroma`.

Instructions:
1. Acknowledge Constraints: Note the assigned `disease_family`. This strictly defines your list of possible diagnoses.
2. Analyze Image: Perform a direct and final analysis of the provided image. Use the initial text report as context, but your diagnosis MUST be based on the visual evidence you see in the image.
3. Critically Evaluate Conflicting Evidence: This is your most important task. Pay close attention to features that might point to different diagnoses within the assigned family. If you observe conflicting evidence (e.g., features typical of BCC like pigment globules AND features typical of SCC like scaling/crusting), you MUST explicitly mention this conflict in your reasoning.
4. Formulate Diagnosis and Differentials:
  -  Your `final_diagnosis` should be the single most likely disease.
  -  Identify the **single next most likely** diagnosis from the same family and place it in the `differential_diagnosis` field.
5. Set Confidence: Your confidence level should reflect the evidence. If the visual features are classic and unambiguous for one diagnosis, confidence is 'High'. If there is significant conflicting evidence or features are poorly defined, your confidence MUST be 'Medium' or 'Low'.

Output Format:
Output ONLY a single JSON object with your final report.
{
  "final_diagnosis": "<Your final specific diagnosis>",
  "confidence": "Low" | "Medium" | "High",
  "differential_diagnosis": "<The single next most likely diagnosis from the same family>",
  "reasoning": "<Your justification for the diagnosis. Explicitly address any conflicting evidence and explain why the final diagnosis is more likely than the differential.>"
}
"""

In [50]:
user_message = [
    {
        "type": "text",
        "text": f"Provide a final diagnosis based on the provided image, the initial report, and the assigned disease family.\n"
    },
    {
        "type": "text",
        "text": "Skin Lesion Image:\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
    },
    {
        "type": "text",
        "text": f"\nInitial Report:\n{json.dumps(image_features, ensure_ascii=False)}"
    },
    {
        "type": "text",
        "text": f"Triage Classification:\n {json.dumps(triage_result, ensure_ascii=False)}"
    }
] 

dx_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    top_p=0.95
)

dx_result = invoke_llm(dx_llm, DIAGNOSIS_PROMPT, user_message)
print(dx_result)

{'final_diagnosis': 'Basal Cell Carcinoma', 'confidence': 'High', 'differential_diagnosis': 'Squamous Cell Carcinoma', 'reasoning': "The lesion is classified within 'Keratinocyte Tumors (Malignant/Pre-malignant)'. Dermoscopic analysis reveals several key features: asymmetrical shape, ill-defined and irregular borders, a large central area of reddish-brown, structureless (homogeneous) pigmentation, and crucially, scattered small, light brown to reddish-brown pigment dots/globules, particularly at the periphery. Additionally, fine, linear, and slightly tortuous vessels are observed. The presence of pigment dots/globules and homogeneous pigmentation are highly characteristic features of pigmented Basal Cell Carcinoma (BCC). While the vascular structures are described as 'fine, linear, and slightly tortuous' rather than classic arborizing vessels, this pattern is not inconsistent with BCC. In contrast, Squamous Cell Carcinoma (SCC), even pigmented variants, typically presents with features

## Add Visual RAG

In [ ]:
# Test Connection
from qdrant_client import QdrantClient

# Connect to local Qdrant instance (Docker)
client = QdrantClient(host="localhost", port=6333)

# Test connection
print(client.get_collections())

collections=[CollectionDescription(name='derm_vkb')]


In [33]:
COLLECTION_NAME = "derm_vkb"
QDRANT_URL = "http://localhost:6333"

In [34]:
import importlib, retrieve_image
importlib.reload(retrieve_image) 

<module 'retrieve_image' from '/Users/thinakonelouangdy/Code/Thesis/Code/experiments/retrieve_image.py'>

In [47]:
from retrieve_image import QdrantBiomedCLIPRetriever

retriever = QdrantBiomedCLIPRetriever(collection=COLLECTION_NAME, qdrant_url=QDRANT_URL)

results = retriever.search(image_path=image_path, k=3)

print(results)

[{'image_path': '/Users/thinakonelouangdy/Code/Thesis/Code/dataset/train/ISIC_0033780.jpg', 'age': 35, 'sex': 'female', 'anatom_site': 'lower extremity', 'diagnosis': 'Dermatofibroma', 'melanocytic': False, 'score': 0.96}, {'image_path': '/Users/thinakonelouangdy/Code/Thesis/Code/dataset/train/ISIC_0033109.jpg', 'age': 45, 'sex': 'female', 'anatom_site': 'anterior torso', 'diagnosis': 'Nevus', 'melanocytic': True, 'score': 0.95}, {'image_path': '/Users/thinakonelouangdy/Code/Thesis/Code/dataset/train/ISIC_0032716.jpg', 'age': 55, 'sex': 'female', 'anatom_site': 'lower extremity', 'diagnosis': 'Melanoma', 'melanocytic': True, 'score': 0.95}]


### Diagnose with Example cases

In [48]:
DX_VRAG_PROMPT = """
You are a specialist AI Dermoscopist. Your task is to provide a final, specific diagnosis by performing a direct visual comparison between a query image and a set of confirmed reference cases.

INPUTS YOU WILL RECEIVE:
-  The original `query_image`.
-  An `initial_report` (containing patient data and a visual summary for context).
-  A `disease_family` classification from the Triage Agent.
-  A list of `reference_cases`, where each case is an image with its confirmed diagnosis from within the assigned family.

INSTRUCTIONS:
1. Analyze Query Image: Briefly describe the key visual features of the `query_image` to establish a baseline for comparison.
2. Perform Comparative Analysis: Perform a head-to-head visual comparison of the `query_image` against EACH of the provided `reference_cases`. For each comparison, you must justify the visual similarities and differences.
3. Synthesize and Decide: Based on your comparative analysis, you must determine:
  - The single best visual match. The diagnosis of this case will be your `final_diagnosis`.
  - The single next-best visual match. The diagnosis of this case will be your `differential_diagnosis`.
4. Set Confidence: Your confidence level should reflect the strength of the best match. If the best match is visually very similar and clearly superior to the others, confidence is 'High'. If the best match is only a fair match, or if it is difficult to distinguish between the top two matches, confidence MUST be 'Medium' or 'Low'.

OUTPUT FORMAT:
Output ONLY a single JSON object with your final report. The reasoning must detail your comparative analysis.
{
  "final_diagnosis": "<The diagnosis of the best-matching reference case>",
  "confidence": "Low" | "Medium" | "High",
  "differential_diagnosis": "<The diagnosis of the second best-matching reference case>",
  "reasoning": {
    "query_summary": "<Your brief visual feature description of the query image.>",
    "comparative_analysis": [
      {
        "reference_diagnosis": "<Diagnosis of Reference Case 1>",
        "justification": "<Your concise detailed reasoning for the visual match, highlighting specific similar or different features.>"
      },
      {
        "reference_diagnosis": "<Diagnosis of Reference Case 2>",
        "justification": "<Your concise detailed reasoning for the visual match...>"
      }
    ],
    "final_synthesis": "<Your concise final conclusion explaining why the chosen reference case is a better visual match than the differential diagnosis.>"
  }
}
"""

In [49]:
case1_image = encode_image(results[0]["image_path"])
case2_image = encode_image(results[1]["image_path"])
case3_image = encode_image(results[2]["image_path"])

user_message = [
    {
        "type": "text",
        "text": f"Provide a final diagnosis based on the provided image, the initial report, and the assigned disease family.\n"
    },
    {
        "type": "text",
        "text": "Skin Lesion Image:\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
    },
    {
        "type": "text",
        "text": f"\nInitial Report:\n{json.dumps(image_features, ensure_ascii=False)}"
    },
    {
        "type": "text",
        "text": f"Triage Classification:\n {json.dumps(triage_result, ensure_ascii=False)}"
    },
    {
        "type": "text",
        "text": "Reference Cases:\n"
    },
    {
        "type": "text",
        "text": f"Reference Case #1: Diagnosis: {results[0]['diagnosis']}, age: {results[0]['age']}, sex: {results[0]['sex']}, lesion_site: {results[0]['anatom_site']}\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{case1_image}" }
    },
    {
        "type": "text",
        "text": f"Reference Case #2: Diagnosis: {results[1]['diagnosis']}, age: {results[1]['age']}, sex: {results[1]['sex']}, lesion_site: {results[1]['anatom_site']}\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{case2_image}" }
    },
    {
        "type": "text",
        "text": f"Reference Case #3: Diagnosis: {results[2]['diagnosis']}, age: {results[2]['age']}, sex: {results[2]['sex']}, lesion_site: {results[2]['anatom_site']}\n"
    },
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{case3_image}" }
    },
] 

dx_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    top_p=0.95
)

dx_result = invoke_llm(dx_llm, DX_VRAG_PROMPT, user_message)
print(dx_result)

{'final_diagnosis': 'Dermatofibroma', 'confidence': 'High', 'differential_diagnosis': 'Nevus', 'reasoning': {'query_summary': 'The query image displays an oval-shaped lesion with an ill-defined border. It features a central reddish-pink discoloration transitioning to light brown and brown homogeneous pigmentation peripherally. A distinct, somewhat irregular whitish structureless area is present centrally within the reddish-pink zone, and the lesion exhibits overall asymmetry.', 'comparative_analysis': [{'reference_diagnosis': 'Dermatofibroma', 'justification': 'Reference Case #1 is an exceptionally strong visual match to the query image. Both images present an oval-shaped lesion with a central whitish structureless area, surrounded by a prominent reddish-pink hue, and peripheral homogeneous light brown to brown pigmentation. The ill-defined borders and overall patternless architecture are virtually identical, strongly suggesting the same diagnosis.'}, {'reference_diagnosis': 'Nevus', '